# SERENE AI — Phase 2: Speech Emotion Model Training
## Fine-Tuning Wav2Vec2 on RAVDESS (Actor-Disjoint Partitions)

**Objective**:
- Fine-tune `facebook/wav2vec2-base` on the official RAVDESS Speech dataset (1,440 clips across 24 actors).
- Preserve all **8 original emotion classes** during model training.
- Enforce an **Actor-Disjoint Partition** (Actors 1–16 train, 17–20 val, 21–24 test) to prevent speaker leakage.
- Evaluate using **Unweighted Average Recall (UAR)** and **Weighted Accuracy (WA)**.
- Implement canonical 7-emotion mapping (`calm` $\to$ `neutral`) for SERENE AI export.
- Save checkpoint to `checkpoints/voice_model/`.

In [ ]:
# 1. Environment Setup & Audio Libraries
!nvidia-smi
!pip install -q transformers torchaudio soundfile scikit-learn pandas tqdm

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForAudioClassification, AutoConfig
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix
from tqdm import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Audio Training Device: {device}")

In [ ]:
# 2. Load Manifest & Actor-Disjoint Split Verification
MANIFEST_PATH = "ravdess_manifest.csv"
df = pd.read_csv(MANIFEST_PATH)

EMOTIONS_8 = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
NUM_CLASSES = 8

print("Split distribution (Actor-Disjoint):")
print(df.groupby(["split", "gender"])["actor_id"].nunique().unstack(fill_value=0))
print("\nTotal clips per split:")
print(df["split"].value_counts())

In [ ]:
# 3. Audio Preprocessing Pipeline (16kHz Mono Resampling & Truncation/Padding)
TARGET_SR = 16000
MAX_DURATION = 3.5  # seconds
MAX_SAMPLES = int(TARGET_SR * MAX_DURATION)

class RAVDESSAudioDataset(Dataset):
    def __init__(self, dataframe, target_sr=16000, max_samples=56000):
        self.df = dataframe.reset_index(drop=True)
        self.target_sr = target_sr
        self.max_samples = max_samples

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]["absolute_path"]
        label = int(self.df.iloc[idx]["emotion_label"])  # 0 to 7
        
        waveform, sr = torchaudio.load(path)
        
        # Convert stereo to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Resample to 16,000 Hz if necessary
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
            
        waveform = waveform.squeeze(0)
        
        # Pad or truncate to fixed sample length
        if len(waveform) > self.max_samples:
            waveform = waveform[:self.max_samples]
        elif len(waveform) < self.max_samples:
            pad_size = self.max_samples - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, pad_size))
            
        return waveform, label

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

train_ds = RAVDESSAudioDataset(train_df, TARGET_SR, MAX_SAMPLES)
val_ds   = RAVDESSAudioDataset(val_df, TARGET_SR, MAX_SAMPLES)
test_ds  = RAVDESSAudioDataset(test_df, TARGET_SR, MAX_SAMPLES)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False)

In [ ]:
# 4. Initialize Wav2Vec2 Model for 8-Class Classification
id2label = {i: name for i, name in enumerate(EMOTIONS_8)}
label2id = {name: i for i, name in enumerate(EMOTIONS_8)}

config = AutoConfig.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    final_dropout=0.3
)

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    config=config
).to(device)

# Freeze early temporal convolution feature encoder layers to preserve speech priors
model.freeze_feature_encoder()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
# 5. Evaluation Protocol: UAR (Unweighted Average Recall) & Weighted Accuracy
def evaluate_voice(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for waveforms, lbls in loader:
            waveforms, lbls = waveforms.to(device), lbls.to(device)
            outputs = model(waveforms)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            
    uar = recall_score(all_labels, all_preds, average='macro')  # Unweighted Average Recall
    wa = accuracy_score(all_labels, all_preds)                  # Weighted Accuracy
    return uar, wa, all_preds, all_labels

print("Ready for Wav2Vec2 training loop on GPU.")

In [ ]:
# 6. Training Loop with UAR-based Checkpointing
best_uar = 0.0
SAVE_DIR = "checkpoints/voice_model"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for waveforms, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        waveforms, lbls = waveforms.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(waveforms)
            loss = criterion(outputs.logits, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        
    scheduler.step()
    val_uar, val_wa, _, _ = evaluate_voice(val_loader)
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f} | Val UAR: {val_uar:.4f} | Val WA: {val_wa:.4f}")
    
    if val_uar > best_uar:
        best_uar = val_uar
        model.save_pretrained(SAVE_DIR)
        print(f"Saved Best Voice Checkpoint to {SAVE_DIR} (Val UAR: {best_uar:.4f})")

# 7. Final Test Evaluation & Canonical 7-Emotion Mapping Verification
test_uar, test_wa, test_preds, test_labels = evaluate_voice(test_loader)
print(f"\n=== Final Test Set (Actor-Disjoint Actors 21-24) ===")
print(f"Unweighted Average Recall (UAR): {test_uar:.4f}")
print(f"Weighted Accuracy (WA):          {test_wa:.4f}")

# Canonical 7-emotion mapping demonstration: calm -> neutral
def map_to_canonical_7(probs_8):
    canonical_map = {
        "angry": "anger", "disgust": "disgust", "fearful": "fear",
        "happy": "joy", "neutral": "neutral", "calm": "neutral",
        "sad": "sadness", "surprised": "surprise"
    }
    canon_probs = {e: 0.0 for e in ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]}
    for i, p in enumerate(probs_8):
        canon_probs[canonical_map[EMOTIONS_8[i]]] += p
    return canon_probs

sample_probs = np.array([0.1, 0.4, 0.05, 0.05, 0.05, 0.05, 0.05, 0.25])
print("\nCanonical 7 Output Example (Calm=0.4 + Neutral=0.1 -> Neutral=0.5):")
print(map_to_canonical_7(sample_probs))